# 🎯 Prompt Engineering - Day 2 Opener

Welcome to Day 2! Today I'm going to show you three techniques that turn a
vague, unreliable prompt into a precision tool.

## Learning Objectives

By the end of this topic you will be able to:

1. Write few-shot prompts that guide the model with labeled examples
2. Use `json_schema` structured outputs to guarantee field names and value types in every response
3. Run a systematic test battery and score your prompt to drive iterative improvement

## What we are building today

The **intent classification layer** of the Barclays Product Knowledge Assistant.

A customer service agent receives a message like:
`"I need help understanding my recent direct debit."`

Before routing that to the right team, we need a reliable classifier that maps
it to one of five categories:

- `account_inquiry`
- `card_services`
- `loans`
- `investments`
- `general`

By end of this topic:

```
BEFORE -> model guesses differently every time, returns free text with no guaranteed format
AFTER  -> model always returns {"intent": "account_inquiry", "confidence": "high", "rationale": "..."}
```

The `classify_intent()` and `classify_with_schema()` functions you build today
are called directly by Topic 5 (Conversation Memory) to route queries.

## Section 0: Environment Setup

Installing dependencies and connecting to SageMaker and the OpenAI API.

In [ ]:
# openai 1.51.0: first version with fully stable json_schema strict mode
# numpy<2 pinned to avoid SageMaker compatibility issues
!pip install -q "openai==1.51.0" "numpy<2"

In [ ]:
import os
import json
import getpass

from openai import OpenAI

import sagemaker
from sagemaker import get_execution_role

# SageMaker session gives us execution role + region
sess = sagemaker.Session()
role = get_execution_role()
print(f"SageMaker region : {sess.boto_region_name}")
print(f"Execution role   : ...{role[-30:]}")

# OpenAI key loaded securely - never hardcoded
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key and press Enter: ")

# Build the client once; it reads OPENAI_API_KEY from the environment automatically
client = OpenAI()
print("OpenAI client ready.")

# The five intent categories we classify every customer query into
INTENT_CATEGORIES = [
    "account_inquiry",
    "card_services",
    "loans",
    "investments",
    "general",
]
print("Intent categories:", INTENT_CATEGORIES)

## What Are We Building Today?

Before we write a single new prompt technique, let me show you the exact problem
we are solving.

A Barclays contact centre receives thousands of queries per day. Before a query
reaches a specialist agent, it needs to be routed. Routing requires classification.

Here are three real-sounding queries:

1. `"Can I increase my overdraft limit?"`
2. `"I think there is a fraudulent charge on my Barclaycard."`
3. `"What are the current ISA interest rates?"`

Each one belongs to a different category. If our prompt is too vague, the model
returns inconsistent labels, adds disclaimers, or formats the answer differently
each time - none of which is good for a downstream router.

That is the problem. Let's fix it, step by step.

## Concept 1: Few-Shot Prompting for Intent Classification

I want to show you something that catches people out when they first try to
build a classifier with an LLM.

Watch what happens when I give the model a zero-shot instruction - just a label list,
no examples. The output looks reasonable at first glance, but run the cell a few
times and watch for inconsistencies.

In [ ]:
# NAIVE ZERO-SHOT DEMO - what NOT to do for production classification
# We give the model the category names but no examples of what each one looks like.
# This works sometimes, but breaks on ambiguous queries.

AMBIGUOUS_QUERIES = [
    "I need help with my direct debit",           # account_inquiry or general?
    "What happens if I miss a loan repayment?",   # loans or general?
    "Can I get a new card if mine is damaged?",   # card_services or general?
]

for query in AMBIGUOUS_QUERIES:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                # No examples - just the category list. The model has to guess.
                "content": (
                    "Classify the following customer query into one of these categories: "
                    + ", ".join(INTENT_CATEGORIES)
                    + ". Reply with the category name only."
                )
            },
            {"role": "user", "content": query}
        ],
        temperature=0.0,
        max_tokens=30
    )
    label = response.choices[0].message.content.strip()
    print(f"Query : {query}")
    print(f"Label : {label}")
    print()

print("Notice: some labels may vary across runs, and the boundary between")
print("'general' and a specific category is unclear without examples.")

### How Few-Shot Prompting Works

<!-- DIAGRAM: few-shot-message-structure - show the messages list with a system message containing 3 labeled examples (Query: ... -> Category: ...) followed by a user message with the live query -->
[View diagram: Few-Shot Message Structure](../../plans/topic_04_prompt_engineering/diagrams/few-shot-message-structure.mmd)
> Diagram coming via /build-diagrams - the file above will contain the Mermaid source once built.

**The core idea**: instead of describing the categories in words, I show the model
examples of (query, label) pairs. The model generalizes from the pattern.

Three things worth knowing:

1. **Order matters (recency bias)**: the model pays more attention to the last
   example in the list. Put the most representative example last.
2. **Label balance**: if 4 of your 5 examples show `account_inquiry`, the model
   will over-predict that category. Balance your examples across categories.
3. **3 to 5 examples** is the sweet spot for classification tasks. More examples
   cost tokens; fewer leave too much ambiguity.

The examples live in the system message, before the user turn. This keeps them
out of the conversation history so they do not shift as turns accumulate.

In [ ]:
# FULL WORKING DEMO - few-shot intent classifier
# I'm going to show you the exact pattern we will use throughout the course.

# Five labeled examples - one per category, balanced, most representative last
FEW_SHOT_EXAMPLES = [
    ("I would like to see my last three statements",         "account_inquiry"),
    ("My Barclaycard was declined at the supermarket",       "card_services"),
    ("What is the early repayment charge on my loan?",       "loans"),
    ("I want to open a stocks and shares ISA",               "investments"),
    ("How do I update my contact details?",                  "general"),
]

def _build_few_shot_system_prompt(examples: list, categories: list) -> str:
    """
    Build a system prompt that contains labeled examples.
    examples: list of (query_text, label) tuples
    categories: list of valid category strings
    """
    # Start with a clear task description and the list of valid categories
    lines = [
        "You are a Barclays customer query classifier.",
        "Classify each customer query into exactly one of these categories:",
        ", ".join(categories),
        "",
        "Here are examples of how to classify queries:",
        "",
    ]
    # Append each labeled example in a consistent (Query / Category) format
    # Consistent formatting helps the model recognize the pattern reliably
    for query_text, label in examples:
        lines.append(f"Query: {query_text}")
        lines.append(f"Category: {label}")
        lines.append("")

    lines.append("Now classify the following query. Reply with the category name only.")
    return "\n".join(lines)


def classify_intent(query: str) -> str:
    """
    Classify a customer query into one of the five Barclays intent categories.
    Returns the category name as a string.
    """
    system_prompt = _build_few_shot_system_prompt(FEW_SHOT_EXAMPLES, INTENT_CATEGORIES)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query}
        ],
        temperature=0.0,   # deterministic - we want the same answer every time
        max_tokens=20      # category name is at most 3 words
    )
    return response.choices[0].message.content.strip().lower()


# Test the classifier on the same ambiguous queries from the naive demo
print("Few-shot classifier results:")
print()
for query in AMBIGUOUS_QUERIES:
    label = classify_intent(query)
    print(f"Query : {query}")
    print(f"Label : {label}")
    print()

## Lab 1: Build Your Own Few-Shot Classifier

Now it's your turn. I want you to write a few-shot system prompt that classifies
queries and test it on three new examples.

**Your task**: Complete the `my_classify_intent()` function below. You will write
your own set of labeled examples and build the system prompt yourself.

**Stretch** (if you finish early): Add a sixth category called `"complaint"` and
extend your examples to cover it. Test whether the model correctly distinguishes
between a complaint (`"I am extremely unhappy with this service"`) and a general query.

**Time**: 15-20 minutes